# Capítulo 08 · Definir y usar funciones

*Traducción y adaptación de [A Whirlwind Tour of Python](https://github.com/jakevdp/WhirlwindTourOfPython)
de Jake VanderPlas (O'Reilly, 2016), publicado con licencia [CC0](https://creativecommons.org/publicdomain/zero/1.0/deed.es).
Adaptación: [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es).*

📖 Lectura complementaria del curso de Python de MiAtlas.cl · se abre en Colab o en JupyterLite, sin GPU.

Hasta ahora, nuestros programas han sido bloques de código simples, de un solo uso.
Una forma de organizar el código Python y hacerlo más legible y reutilizable es separar las piezas útiles en
*funciones* reutilizables.
Aquí veremos dos formas de crear funciones: la instrucción `def`, útil para cualquier tipo de función, y la expresión
`lambda`, útil para crear funciones anónimas y cortas.

## Usar funciones

Las funciones son grupos de código que tienen un nombre y que se pueden llamar con paréntesis.
Ya hemos visto funciones antes. Por ejemplo, `print` es una función:

In [ ]:
print('abc')

Aquí `print` es el nombre de la función y `'abc'` es el *argumento* de la función.

Además de los argumentos, hay *argumentos con nombre*, que se indican por su nombre.
Uno de los argumentos con nombre de la función `print()` es `sep`, que indica qué carácter o caracteres se usan para
separar varios elementos:

In [ ]:
print(1, 2, 3)

In [ ]:
print(1, 2, 3, sep='--')

Cuando se usan argumentos sin nombre junto con argumentos con nombre, los argumentos con nombre deben ir al final.

## Definir funciones

Las funciones se vuelven aún más útiles cuando empezamos a definir las nuestras, organizando funcionalidades que se
usarán en varios lugares.
En Python, las funciones se definen con la instrucción `def`.
Por ejemplo, podemos encapsular una versión del código de la sucesión de Fibonacci del capítulo anterior así:

In [ ]:
def fibonacci(N):
    L = []
    a, b = 0, 1
    while len(L) < N:
        a, b = b, a + b
        L.append(a)
    return L

Ahora tenemos una función llamada `fibonacci` que recibe un único argumento `N`, hace algo con él y *devuelve*
(`return`) un valor; en este caso, una lista con los primeros `N` números de Fibonacci:

In [ ]:
fibonacci(10)

Si conoces lenguajes con tipado estático como C, notarás de inmediato que no hay información de tipos asociada a las
entradas ni a las salidas de la función.
Las funciones de Python pueden devolver cualquier objeto de Python, simple o compuesto, lo que hace que construcciones
difíciles en otros lenguajes sean directas en Python.

Por ejemplo, para devolver varios valores basta con ponerlos en una tupla, que se indica con comas:

In [ ]:
def real_imag_conj(val):
    return val.real, val.imag, val.conjugate()

r, i, c = real_imag_conj(3 + 4j)
print(r, i, c)

> 📝 **Nota de MiAtlas.cl:** desde Python 3.5 puedes agregar *anotaciones de tipo* opcionales a una función. Python
> no las verifica al ejecutar, pero sirven como documentación y las usan los editores y herramientas como `mypy`:
>
> ```python
> def fibonacci(N: int) -> list[int]:
>     ...
> ```
>
> La forma `list[int]` (en lugar de `List[int]` del módulo `typing`) funciona desde Python 3.9.

## Valores por omisión de los argumentos

A menudo, al definir una función, hay ciertos valores que queremos que la función use *la mayoría* de las veces,
pero también queremos darle cierta flexibilidad a quien la usa.
Para eso podemos dar *valores por omisión* a los argumentos.
Retomemos la función `fibonacci`. ¿Y si quisiéramos que se pudiera jugar con los valores iniciales?
Podríamos hacerlo así:

In [ ]:
def fibonacci(N, a=0, b=1):
    L = []
    while len(L) < N:
        a, b = b, a + b
        L.append(a)
    return L

Con un único argumento, el resultado de la llamada es idéntico al de antes:

In [ ]:
fibonacci(10)

Pero ahora podemos usar la función para explorar cosas nuevas, como el efecto de otros valores iniciales:

In [ ]:
fibonacci(10, 0, 2)

Si quieres, los valores también se pueden indicar por su nombre; en ese caso, el orden de los argumentos con nombre no
importa:

In [ ]:
fibonacci(10, b=3, a=1)

> 📝 **Nota de MiAtlas.cl:** el valor por omisión se calcula **una sola vez**, al definir la función. Por eso no
> conviene usar un objeto mutable (como `[]` o `{}`) como valor por omisión: todas las llamadas compartirían el mismo
> objeto. La costumbre es usar `None` y crear el objeto dentro de la función:
>
> ```python
> def agregar(elemento, lista=None):
>     if lista is None:
>         lista = []
>     lista.append(elemento)
>     return lista
> ```

## `*args` y `**kwargs`: argumentos flexibles

A veces quieres escribir una función sin saber de antemano cuántos argumentos le van a pasar.
En ese caso puedes usar las formas especiales `*args` y `**kwargs` para capturar todos los argumentos que lleguen.
Este es un ejemplo:

In [ ]:
def atrapa_todo(*args, **kwargs):
    print("args =", args)
    print("kwargs =", kwargs)

In [ ]:
atrapa_todo(1, 2, 3, a=4, b=5)

In [ ]:
atrapa_todo('a', clave=2)

Aquí lo importante no son los nombres `args` y `kwargs`, sino los caracteres `*` que los preceden.
`args` y `kwargs` son solo los nombres que se usan por convención, abreviaturas de *arguments* (argumentos) y
*keyword arguments* (argumentos con nombre).
La diferencia que importa son los asteriscos: un solo `*` antes de una variable significa «expándela como una
secuencia», y un doble `**` significa «expándela como un diccionario».
De hecho, esta sintaxis se puede usar no solo al definir la función, ¡también al llamarla!

In [ ]:
entradas = (1, 2, 3)
con_nombre = {'pi': 3.14}

atrapa_todo(*entradas, **con_nombre)

> 📝 **Nota de MiAtlas.cl:** desde Python 3.8 también puedes exigir cómo se pasa cada parámetro. Los que van antes de
> una `/` solo se pueden pasar por posición, y los que van después de un `*` solo se pueden pasar por nombre:
>
> ```python
> def f(a, b, /, c, *, d):
>     ...
>
> f(1, 2, 3, d=4)     # válido
> f(1, 2, c=3, d=4)   # válido
> # f(a=1, b=2, c=3, d=4) lanzaría TypeError: a y b son solo posicionales
> ```

## Funciones anónimas (`lambda`)

Antes vimos rápidamente la forma más común de definir funciones, la instrucción `def`.
Probablemente te encontrarás con otra forma de definir funciones cortas, de un solo uso, con la expresión `lambda`.
Se ve más o menos así:

In [ ]:
suma = lambda x, y: x + y
suma(1, 2)

Esta función lambda equivale, a grandes rasgos, a:

In [ ]:
def suma(x, y):
    return x + y

¿Para qué querrías usar algo así?
Principalmente, porque en Python *todo es un objeto*, ¡incluso las funciones!
Eso significa que las funciones se pueden pasar como argumentos a otras funciones.

Como ejemplo, supongamos que tenemos unos datos guardados en una lista de diccionarios:

In [ ]:
datos = [{'nombre': 'Guido', 'apellido': 'Van Rossum', 'nacimiento': 1956},
         {'nombre': 'Grace', 'apellido': 'Hopper',     'nacimiento': 1906},
         {'nombre': 'Alan',  'apellido': 'Turing',     'nacimiento': 1912}]

Ahora supongamos que queremos ordenar estos datos.
Python tiene una función `sorted` que hace justamente eso:

In [ ]:
sorted([2, 4, 3, 5, 1, 6])

Pero los diccionarios no se pueden ordenar entre sí: necesitamos una forma de decirle a la función *cómo* ordenar
nuestros datos.
Podemos hacerlo con el argumento `key`, una función que, dado un elemento, devuelve la clave de orden de ese elemento:

In [ ]:
# ordenar alfabéticamente por nombre
sorted(datos, key=lambda item: item['nombre'])

In [ ]:
# ordenar por año de nacimiento
sorted(datos, key=lambda item: item['nacimiento'])

Aunque estas funciones clave se podrían crear con la sintaxis normal de `def`, la sintaxis `lambda` es cómoda para
funciones cortas de un solo uso como estas.

## Pruébalo tú

Escribe una función que reciba **cualquier cantidad de números** (usa `*numeros`) y devuelva su **promedio**.
Si no recibe ningún número, debe devolver `0`. Puedes definirla con `def` o con `lambda`; después, guarda la función
(sin llamarla, sin paréntesis) en `resultado`.

In [ ]:
resultado = None

# Escribe aquí tu función y luego: resultado = nombre_de_tu_funcion

In [ ]:
# Comprobación
if resultado is None:
    print("Aún no lo resuelves.")
elif callable(resultado) and resultado(2, 4, 6) == 4 and resultado(5) == 5 and resultado() == 0:
    print("✅ ¡Correcto!")
else:
    print("❌ Revisa tu respuesta")

**Sigue en MiAtlas.cl:** tema «Funciones y parámetros» · «Alcance, clausuras y lambdas».

**Original:** [08-Defining-Functions.ipynb](https://github.com/jakevdp/WhirlwindTourOfPython/blob/master/08-Defining-Functions.ipynb)